# Paper figures

Reproduces every data panel and in-text statistic of *Extracting Search Trees from LLM Reasoning Traces Reveals
Myopic Planning* from the files in `data/` and `results/`. Each panel is saved as a PDF in `figures/`.
Panels drawn by hand or composed in the layout (1B, the text boxes of 1C, 3B, 4A) are not reproduced.

| Section | Content | Inputs |
|---|---|---|
| Fig 1 | Board positions (A, C); extracted search tree (D) | `data/game_trees.pkl` |
| Fig 2, §3.2 | Search effort vs win rate; breadth vs depth; depth/breadth regression | `data/game_trees.pkl` |
| Fig 3, §4.2 | Heuristic features; model comparison; depth harm; candidate gain; C vs win rate | `results/fits/pattern/` |
| Fig 4B | Intervention move change rates | `results/intervention/` |
| Fig S1 | Nodes per depth, per model | `data/game_trees.pkl` |
| Fig S2 | Model recovery | `results/recovery/` |
| Fig S3 | Normalized feature weights vs win rate | `results/fits/pattern/` |
| Fig S4 | Model comparison with the window-count features | `results/fits/windowcount/` |

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, Circle
from scipy.stats import pearsonr, t as t_dist

from fiar.models import TRACE_ACCESSIBLE, FITTED_MODELS
from fiar.render import render_tree

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

RESULTS_DIR = Path('results')
FIGDIR = Path('figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

def save(fig, name, **kw):
    kw.setdefault('facecolor', 'white')
    fig.savefig(FIGDIR / f'{name}.pdf', **kw)
    print(f'Saved: {FIGDIR / name}.pdf')

In [ ]:
# ── Shared style ───────────────────────────────────────────────────────────
def is_open_source(name):
    # the 14 models whose full reasoning traces are accessible
    return name in TRACE_ACCESSIBLE

def family_color(name):
    n = name.lower()
    if 'oss'      in n: return '#009900'
    if 'gpt' in n or 'o3' in n: return '#009900'
    if 'deepseek' in n: return '#1f77b4'
    if 'qwen'     in n: return '#cc00ff'
    if 'kimi'     in n: return '#e6b800'
    if 'glm'      in n: return '#ff3333'
    return '#3A3939'   # llama / other

def sig_stars(p):
    return '**' if p < 0.01 else ('*' if p < 0.05 else ('.' if p < 0.1 else ''))

# Dashed lines connecting models within a family
LINE_GROUPS = [
    ('#009900', ['gpt-oss-120b-medium', 'gpt-oss-120b-high']),
    ('#1f77b4', ['deepseek-v3.1', 'deepseek-v3.1-thinking', 'deepseek-v3.2-thinking']),
    ('#e6b800', ['kimi-k2-instruct', 'kimi-k2-thinking']),
    ('#ff3333', ['glm-4.5-thinking', 'glm-4.6']),
]

def draw_family_lines(ax, coords):
    # coords: {model_name: (x, y)}
    for color, group in LINE_GROUPS:
        present = [m for m in group if m in coords]
        if len(present) >= 2:
            ax.plot([coords[m][0] for m in present], [coords[m][1] for m in present],
                    color=color, linestyle='--', linewidth=1.5, alpha=0.7, zorder=0)

def label_points(ax, xs, ys, names, offsets, fontsize=6.5):
    for x, y, name in zip(xs, ys, names):
        ax.annotate(name, (x, y), xytext=offsets.get(name, (6, 2)),
                    textcoords='offset points', fontsize=fontsize)

## Load data

- `turns`: one row per turn with the extracted tree and its statistics (`data/game_trees.pkl`)
- `win_rate`: per-model win rate, averaged within game then across games
- `wide`: per-model NLL / accuracy for the full-tree, myopic, no-tree and discount (γ) fits
- `myopic_w`: fitted myopic-model feature weights

In [ ]:
turns = pd.read_pickle('data/game_trees.pkl')

win_rate = (
    turns.groupby(['game_path', 'model_names'])['outcome_perspective'].mean()
    .reset_index().groupby('model_names')['outcome_perspective'].mean()
)

In [ ]:
# ── Cognitive-model fits (full / myopic / no-tree / discount), pattern features ──
VARIANTS = ['full', 'myopic', 'notree', 'gamma']

def load_fits(basis):
    records = []
    for model in FITTED_MODELS:
        for variant in VARIANTS:
            d = json.load(open(RESULTS_DIR / 'fits' / basis / f'{model}_{variant}.json'))
            records.append({'model': model, 'variant': variant, 'n_samples': d['n_samples'],
                            'nll': d['nll_per_sample'], 'accuracy': d['accuracy'],
                            'weights': d['weights'], 'C': d['C'], 'gamma': d['gamma']})
    fits = pd.DataFrame(records)
    wide = fits.pivot(index='model', columns='variant', values=['nll', 'accuracy'])
    wide.columns = [f'{metric}_{var}' for metric, var in wide.columns]
    return fits, wide

fits, wide = load_fits('pattern')
print(f'{len(wide)} models')
wide.round(3)

## Figure 1: Game setup and search tree extraction

Boards for panel A ("Current board state") and panel C ("Black player's turn", "White player's turn").
The cursor icons, prompt boxes and robot icons were added in the layout.

In [ ]:
def draw_board(white, black, lines=(), legend=None, name=None):
    fig, ax = plt.subplots(figsize=(8.12, 3.76), dpi=100)
    fig.patch.set_facecolor('#ffffff')
    ax.set_facecolor('#dddddd')

    ncols, nrows = 9, 4
    grid_lw = 2.3
    ax.add_patch(Rectangle((0, 0), ncols, nrows, facecolor='#9f9f9f', edgecolor='black', linewidth=grid_lw))
    for x in range(1, ncols):
        ax.plot([x, x], [0, nrows], color='black', lw=grid_lw)
    for y in range(1, nrows):
        ax.plot([0, ncols], [y, y], color='black', lw=grid_lw)

    r = 0.42
    for x, y in white:
        ax.add_patch(Circle((x, y), r, facecolor='white', edgecolor='none'))
    for x, y in black:
        ax.add_patch(Circle((x, y), r, facecolor='black', edgecolor='none'))

    for (x0, y0), (x1, y1), color in lines:
        ax.plot([x0, x1], [y0, y1], color=color, lw=3.8, solid_capstyle='round')

    ax.set_xlim(-0.1, ncols + 0.1)
    ax.set_ylim(-0.1, nrows + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    if legend:
        ax.legend(handles=legend, loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=2,
                  frameon=False, fontsize=18)
    plt.tight_layout(pad=0)
    if name:
        save(fig, name, format='pdf', bbox_inches='tight', pad_inches=0.05)
    plt.show()

# Piece centres in board coordinates: x = col + 0.5, y = (3 - row) + 0.5
WHITE = [(1.5, 3.5), (1.5, 1.5), (3.5, 2.5), (5.5, 2.5), (4.5, 1.5)]
BLACK = [(2.5, 3.5), (3.5, 3.5), (2.5, 2.5), (2.5, 1.5), (3.5, 1.5)]
BLACK_MOVE = (2.5, 1.5)   # black plays (2, 2)
WHITE_MOVE = (2.5, 0.5)   # white replies (3, 2)

In [ ]:
# Fig 1A: current board state (before black's move)
draw_board(WHITE, [b for b in BLACK if b != BLACK_MOVE], name='fig1A_board_current')

In [ ]:
# Fig 1C (left): after black plays (2, 2)
draw_board(WHITE, BLACK, name='fig1C_board_black_turn')

In [ ]:
# Fig 1C (right): after white replies (3, 2)
draw_board(WHITE + [WHITE_MOVE], BLACK, name='fig1C_board_white_turn')

### Fig 1D: extracted search tree

Rendered with graphviz (`fiar.render`). As the caption says, this tree is illustrative and is not from the board
position in panel A.

In [ ]:
FIG1D_ROW = 17227   # qwen3-next-80b-a3b-thinking turn with 15 first-ply candidates
row = turns.loc[FIG1D_ROW]
print('model:', row['model_names'], '| total_nodes:', row['total_nodes'])
render_tree(row['trees'], row['board_state'], filename=str(FIGDIR / 'fig1D_search_tree'))

## Figure 2: Planning effort and game performance

### Fig 2A: winning rate vs tree size

The points are plotted against mean tree size, while $r$ in the title is computed against log(mean tree size + 1),
as in the original analysis.

In [ ]:
model_stats = turns.groupby('model_names').agg(total_nodes=('total_nodes', 'mean')).reset_index()
model_stats['rating'] = model_stats['model_names'].map(win_rate)
model_stats['log_search_effort'] = np.log(model_stats['total_nodes'] + 1)
model_stats = model_stats[model_stats['model_names'].map(is_open_source)]

r, p = pearsonr(model_stats['log_search_effort'], model_stats['rating'])

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(model_stats['total_nodes'], model_stats['rating'],
           c=[family_color(m) for m in model_stats['model_names']], s=75, alpha=0.9, edgecolors='none')
label_points(ax, model_stats['total_nodes'], model_stats['rating'], model_stats['model_names'],
             offsets={'qwen3-235b-a22b-thinking': (-90, 6), 'gpt-oss-120b-medium': (4, -7)})
draw_family_lines(ax, {row['model_names']: (row['total_nodes'], row['rating'])
                       for _, row in model_stats.iterrows()})

ax.set_xlim(-1.2, 23)
ax.set_ylim(0, 0.85)
ax.set_xlabel('Tree size', fontsize=11)
ax.set_ylabel('Win rate', fontsize=11)
ax.set_title(rf'$r$ = {r:.2f}, $p$ = {p:.3f} {sig_stars(p)}', fontsize=11)
plt.tight_layout()
save(fig, 'fig2A_winrate_vs_tree_size')
plt.show()

### Fig 2B: breadth vs depth; Fig 2C: winning rate vs breadth-depth ratio

In [ ]:
# Per-model tree shape, over successful moves only
ts = turns[turns['model_names'].map(is_open_source) & turns['move_success']].copy()
ts['model'] = ts['model_names']
ts['depth_breadth_ratio'] = ts['max_ply'] / ts['n_root_moves'].replace(0, np.nan)

shape = ts.groupby('model').agg(
    mean_max_depth=('max_ply', 'mean'),
    mean_breadth=('n_root_moves', 'mean'),
    mean_depth_breadth=('depth_breadth_ratio', 'mean'),
)
shape['win_rate'] = (
    ts.groupby(['game_path', 'model'])['outcome_perspective'].mean()
    .reset_index().groupby('model')['outcome_perspective'].mean()
)
shape.round(3)

In [ ]:
# Fig 2B
r, p = pearsonr(shape['mean_max_depth'], shape['mean_breadth'])

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(shape['mean_max_depth'], shape['mean_breadth'],
           c=[family_color(m) for m in shape.index], s=65, alpha=0.9, zorder=3, edgecolors='none')
label_points(ax, shape['mean_max_depth'], shape['mean_breadth'], shape.index,
             offsets={'deepseek-v3.1-thinking': (-82, 2), 'kimi-k2-thinking': (-59, 2)})
draw_family_lines(ax, {m: (row['mean_max_depth'], row['mean_breadth']) for m, row in shape.iterrows()})

ax.set_xlim(0.7, 3.95)
ax.set_xlabel('Depth (max ply)', fontsize=11)
ax.set_ylabel('Breadth (# first-ply candidates)', fontsize=11)
ax.set_title(rf'$r$ = {r:.2f}, $p$ = {p:.3f} {sig_stars(p)}', fontsize=11)
plt.tight_layout()
save(fig, 'fig2B_breadth_vs_depth')
plt.show()

In [ ]:
# Fig 2C: breadth-depth ratio = 1 / mean(max_ply / n_root_moves)
breadth_depth = 1 / shape['mean_depth_breadth']
r, p = pearsonr(breadth_depth, shape['win_rate'])

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(breadth_depth, shape['win_rate'],
           c=[family_color(m) for m in shape.index], s=65, alpha=0.9, zorder=3, edgecolors='none')
label_points(ax, breadth_depth, shape['win_rate'], shape.index,
             offsets={'qwen3-235b-a22b-thinking': (6, -6), 'qwen3-next-80b-a3b-thinking': (6, -6),
                      'gpt-oss-120b-medium': (-77, 2), 'glm-4.6': (-30, 2)})

ax.set_xlim(0.8, 4.3)
ax.set_ylim(0, 0.85)
ax.set_xlabel('Breadth-depth ratio', fontsize=11)
ax.set_ylabel('Win rate', fontsize=11)
ax.set_title(rf'$r$ = {r:.2f}, $p$ = {p:.3f} {sig_stars(p)}', fontsize=11)
plt.tight_layout()
save(fig, 'fig2C_winrate_vs_breadth_depth_ratio')
plt.show()

### Section 3.2: search depth and the depth/breadth regression

Per-model mean depth (max ply) and breadth over all turns of the 14 trace-accessible models, and an OLS regression
of win rate on both.

In [ ]:
per_model = turns[turns['model_names'].map(is_open_source)].groupby('model_names').agg(
    depth=('max_ply', 'mean'), breadth=('n_root_moves', 'mean'))
per_model['win_rate'] = per_model.index.map(win_rate)
d = per_model['depth']
print(f'Mean depth across models: {d.min():.2f}-{d.max():.2f} plies ({d.mean():.2f} ± {d.std():.2f})')

X = np.column_stack([np.ones(len(per_model)), per_model['depth'], per_model['breadth']])
y = per_model['win_rate'].to_numpy()
beta = np.linalg.lstsq(X, y, rcond=None)[0]
dof = len(y) - X.shape[1]
se = np.sqrt(np.diag(((y - X @ beta) ** 2).sum() / dof * np.linalg.inv(X.T @ X)))
p = 2 * t_dist.sf(np.abs(beta / se), dof)
for name, b, pv in zip(['depth', 'breadth'], beta[1:], p[1:]):
    print(f'{name}: beta = {b:+.3f}, p = {pv:.4f}')

## Figure 3: Predicting moves with cognitive models

### Fig 3A: heuristic features on the board

In [ ]:
def draw_board(white, black, lines=(), legend=None, name=None):
    fig, ax = plt.subplots(figsize=(8.12, 3.76), dpi=100)
    fig.patch.set_facecolor('#ffffff')
    ax.set_facecolor('#dddddd')

    ncols, nrows = 9, 4
    grid_lw = 2.3
    ax.add_patch(Rectangle((0, 0), ncols, nrows, facecolor='#9f9f9f', edgecolor='black', linewidth=grid_lw))
    for x in range(1, ncols):
        ax.plot([x, x], [0, nrows], color='black', lw=grid_lw)
    for y in range(1, nrows):
        ax.plot([0, ncols], [y, y], color='black', lw=grid_lw)

    r = 0.42
    for x, y in white:
        ax.add_patch(Circle((x, y), r, facecolor='white', edgecolor='none'))
    for x, y in black:
        ax.add_patch(Circle((x, y), r, facecolor='black', edgecolor='none'))

    for (x0, y0), (x1, y1), color in lines:
        ax.plot([x0, x1], [y0, y1], color=color, lw=3.8, solid_capstyle='round')

    ax.set_xlim(-0.1, ncols + 0.1)
    ax.set_ylim(-0.1, nrows + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    if legend:
        ax.legend(handles=legend, loc='lower center', bbox_to_anchor=(0.55, -1.4), ncol=1,
                  frameon=False, fontsize=20)
    plt.tight_layout(pad=0)
    if name:
        save(fig, name, format='pdf', bbox_inches='tight', pad_inches=0.05)
    plt.show()

FEAT_BLUE, FEAT_ORANGE, FEAT_PURPLE = '#2f49a7', '#f06a24', '#a24db5'
feature_lines = [
    ((1.5, 0.5), (1.5, 3.5), FEAT_BLUE),
    ((2.5, 0.5), (2.5, 3.5), FEAT_ORANGE),
    ((3.5, 2.5), (6.5, 2.5), FEAT_BLUE),
    ((2.5, 3.5), (5.5, 3.5), FEAT_PURPLE),
    ((3.5, 0.5), (6.5, 3.5), FEAT_PURPLE),
]
feature_legend = [
    mlines.Line2D([], [], color=FEAT_BLUE,   lw=2.5, label='unconnected two-in-a-row'),
    mlines.Line2D([], [], color=FEAT_PURPLE, lw=2.5, label='connected two-in-a-row'),
    mlines.Line2D([], [], color=FEAT_ORANGE, lw=2.5, label='three-in-a-row'),
]
draw_board(WHITE, BLACK, lines=feature_lines, legend=feature_legend, name='fig3A_features')

### Fig 3C: NLL per sample; Fig 3D: prediction accuracy

Models are sorted by myopic accuracy in both panels. In panel D the Discount bar is set to the myopic accuracy, as in
the original analysis notebook (the fitted γ is ≈0, so the discount model reduces to the myopic one); delete that line
to plot the discount fit's own accuracy.

In [ ]:
# VAR_COLORS = [('notree', '#cc44ff'), ('full', '#1f77b4'), ('myopic', '#ff9900'), ('gamma', '#44ff88')]
VAR_COLORS = [('notree', "#d255ff"), ('full', "#38a2ed"), ('myopic', "#3dee7e"), ('gamma', "#ffab2e")]
VAR_LABELS = {'notree': 'No-tree', 'full': 'Full-tree', 'myopic': 'Myopic', 'gamma': 'Discount'}
BAR_HEIGHT = 0.2
OFFSETS = [-1.5, -0.5, 0.5, 1.5]

def variant_bars(metric, xlabel, xlim=None, legend=False):
    order = wide.sort_values('accuracy_myopic').index.tolist()
    y = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(5.7, 4.5))
    for (variant, color), off in zip(VAR_COLORS, OFFSETS):
        ax.barh(y + off * BAR_HEIGHT, [wide.loc[m, f'{metric}_{variant}'] for m in order], BAR_HEIGHT,
                label=VAR_LABELS[variant], color=color, alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(order, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=11)
    if xlim:
        ax.set_xlim(*xlim)
    if legend:
        ax.legend(loc='lower right', frameon=False, fontsize=8)
    plt.tight_layout()
    return fig

In [ ]:
# Fig 3C
fig = variant_bars('nll', 'Negative log likelihood per sample')
save(fig, 'fig3C_nll', bbox_inches='tight')
plt.show()

In [ ]:
# Fig 3D
wide['accuracy_gamma'] = wide['accuracy_myopic']   # see note above
fig = variant_bars('accuracy', 'Prediction accuracy', xlim=(0, 0.7), legend=False)
save(fig, 'fig3D_accuracy', bbox_inches='tight')
plt.show()

### Section 4.2: turns where only one of the full-tree and myopic models is correct

Each model predicts the highest-valued candidate under its fitted parameters.

In [ ]:
from fiar.fit import build_dataset, load_turns, make_myopic
from fiar.minimax import root_action_values
from fiar.features import BASES

def predicted_moves(dataset, fit):
    value = BASES['pattern'].value
    preds = []
    for s in dataset:
        q = root_action_values(s['tree'], value, np.array(fit['weights']), fit['C'], 1.0)
        preds.append(max(q, key=q.get))
    return preds

myopic_only = full_only = 0
for model in FITTED_MODELS:
    dataset = build_dataset(load_turns('data/game_trees.pkl', model))
    fit = fits[fits['model'] == model].set_index('variant')
    pred_full = predicted_moves(dataset, fit.loc['full'])
    pred_myopic = predicted_moves(make_myopic(dataset), fit.loc['myopic'])
    for s, pf, pm in zip(dataset, pred_full, pred_myopic):
        if pf != pm:
            myopic_only += pm == s['observed_move']
            full_only += pf == s['observed_move']
print(f'Uniquely correct: myopic {myopic_only} turns, full-tree {full_only} turns')
print('Fitted discount factors:', fits[fits['variant'] == 'gamma'].set_index('model')['gamma'].round(4).to_dict())

### Fig 3E: depth harm vs depth; Fig 3F: candidate gain vs breadth

- Depth harm = NLL(full) − NLL(myopic)
- Candidate gain = NLL(no-tree) − NLL(myopic)

In [ ]:
nll_deltas = pd.DataFrame({
    'depth_harm':     wide['nll_full']   - wide['nll_myopic'],
    'candidate_gain': wide['nll_notree'] - wide['nll_myopic'],
})
search_stats = (
    turns.groupby('model_names').agg(mean_max_ply=('max_ply', 'mean'), mean_n_root=('n_root_moves', 'mean'))
)
nll_deltas = nll_deltas.join(search_stats)
nll_deltas.round(3)

In [ ]:
# Fig 3E
sub = nll_deltas[['mean_max_ply', 'depth_harm']].dropna()
r, p = pearsonr(sub['mean_max_ply'], sub['depth_harm'])

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(sub['mean_max_ply'], sub['depth_harm'], c=[family_color(m) for m in sub.index],
           s=70, alpha=0.9, edgecolors='none', zorder=3)
label_points(ax, sub['mean_max_ply'], sub['depth_harm'], sub.index,
             offsets={'kimi-k2-instruct': (6, -5), 'gpt-oss-120b-medium': (6, -7), 'deepseek-v3.1-thinking': (-82, -6)})
draw_family_lines(ax, {m: (row['mean_max_ply'], row['depth_harm']) for m, row in sub.iterrows()})

ax.set_xlabel('Depth (max ply)', fontsize=11)
ax.set_ylabel('Depth harm (NLL: full − myopic)', fontsize=11)
ax.set_title(rf'$r$ = {r:.2f}, $p$ = {p:.3f} {sig_stars(p)}', fontsize=11)
ax.set_xlim(0.85, 3.8)
ax.set_ylim(0, 0.55)
plt.tight_layout()
save(fig, 'fig3E_depth_harm_vs_depth')
plt.show()

In [ ]:
# Fig 3F
sub = nll_deltas[['mean_n_root', 'candidate_gain']].dropna()
r, p = pearsonr(sub['mean_n_root'], sub['candidate_gain'])
print(f'r = {r:.3f}, p = {p:.2g}')

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(sub['mean_n_root'], sub['candidate_gain'], c=[family_color(m) for m in sub.index],
           s=70, alpha=0.9, edgecolors='none', zorder=3)
label_points(ax, sub['mean_n_root'], sub['candidate_gain'], sub.index,
             offsets={'qwen3-235b-a22b-thinking': (-94, 2), 'deepseek-v3.2-thinking': (-83, 2), 'glm-4.6': (-29, 2)})
draw_family_lines(ax, {m: (row['mean_n_root'], row['candidate_gain']) for m, row in sub.iterrows()})

ax.set_xlim(1, 12.8)
ax.set_ylim(0, 1.35)
ax.set_xlabel('Breadth (# first-ply candidates)', fontsize=11)
ax.set_ylabel('Candidate gain (NLL: no-tree − myopic)', fontsize=11)
ax.set_title(rf'$r$ = {r:+.2f}, $p$ < .001 {sig_stars(p)}', fontsize=11)   # p hard-coded as "< .001"
plt.tight_layout()
save(fig, 'fig3F_candidate_gain_vs_breadth')
plt.show()

### Fig 3G: winning rate vs offensive bias C

C and the feature weights below come from the **myopic** fit. llama-v3.3-70b is excluded.

In [ ]:
myopic = fits[(fits['variant'] == 'myopic') & (fits['model'] != 'llama-v3.3-70b-instruct')].copy()
W_COLS = ['w_centre', 'w_c2', 'w_uc2', 'w_three', 'w_four']
myopic[W_COLS] = pd.DataFrame(myopic['weights'].tolist(), index=myopic.index)
for col in W_COLS[:-1]:
    myopic[f'{col}_norm'] = myopic[col] / myopic['w_four']   # normalize by the four-in-a-row weight
myopic['win_rate'] = myopic['model'].map(win_rate)
myopic_w = myopic.set_index('model')
myopic_w[['C'] + W_COLS + ['win_rate']].round(3)

In [ ]:
r, p = pearsonr(myopic_w['C'], myopic_w['win_rate'])
print(f'r = {r:.3f}, p = {p:.2g}')

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(myopic_w['C'], myopic_w['win_rate'], c=[family_color(m) for m in myopic_w.index],
           s=80, alpha=0.9, edgecolors='none', zorder=3)
label_points(ax, myopic_w['C'], myopic_w['win_rate'], myopic_w.index,
             offsets={'qwen3-235b-a22b-thinking': (-94, -6), 'deepseek-v3.1': (6, -6), 'gpt-oss-120b-medium': (6, -6)})
ax.axvline(1, color='gray', lw=0.8, ls='--')

ax.set_xlim(0, 3.5)
ax.set_ylim(0.11, 0.85)
ax.set_xlabel('C (defensive bias)', fontsize=11)
ax.set_ylabel('Win rate', fontsize=11)
ax.set_title(rf'$r$ = {r:.2f}, $p$ < .001 {sig_stars(p)}', fontsize=11)   # p hard-coded as "< .001"
plt.tight_layout()
save(fig, 'fig3G_winrate_vs_offensive_bias')
plt.show()

## Figure 4B: Causal intervention

Move change rate after each pruning condition, for the three models. Orange is the chosen branch; gray is the
largest unchosen branch.

In [ ]:
CHOSEN, UNCHOSEN = '#e07b54', '#aaaaaa'
INTERVENTION_MODELS = ['qwen3-next-80b-a3b-thinking', 'qwen3-235b-a22b-thinking', 'kimi-k2-thinking']
labels = ['Remove\nfinal decision', 'Remove\nfinal decision\n+whole branch',
          'Add back\ndepth-1 only', 'Add back\ndepth-1 only\n+depth-2 only'][::-1]
pairs = [('fd', None), ('fd_branch_comp', 'fd_branch_comp_ctrl'),
         ('bc_minus_d0only', 'bc_minus_d0only_ctrl'),
         ('bc_minus_d0_and_deep1plus', 'bc_minus_d0_and_deep1plus_ctrl')][::-1]

w = 0.35
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True)
for ax, model in zip(axes, INTERVENTION_MODELS):
    rate = lambda s: pd.read_csv(RESULTS_DIR / 'intervention' / model / f'results_{s}.csv')['move_changed'].mean() * 100
    for i, (s, ctrl) in enumerate(pairs):
        bars = [(i, rate(s), CHOSEN)] if ctrl is None else \
               [(i + w / 2, rate(s), CHOSEN), (i - w / 2, rate(ctrl), UNCHOSEN)]
        for y, v, color in bars:
            ax.barh(y, v, w, color=color)
            ax.text(v + 1, y, f'{v:.1f}%', va='center', fontsize=10)
    ax.set_title(model, fontsize=11)
    ax.set_xlim(0, 70)
    ax.set_xticks([0, 20, 40, 60])
    ax.tick_params(axis='y', length=0)
axes[0].set_yticks(np.arange(len(labels)))
axes[0].set_yticklabels(labels, fontsize=9)
axes[1].set_xlabel('Move change rate (%)', fontsize=11)
fig.legend(handles=[mpatches.Patch(color=CHOSEN, label='Chosen branch'),
                    mpatches.Patch(color=UNCHOSEN, label='Unchosen branch')],
           fontsize=8, loc='lower right', bbox_to_anchor=(0.99, 0.22), frameon=False)
plt.tight_layout()
save(fig, 'fig4B_intervention')
plt.show()

## Figure S1: Distribution of node depth per model

Mean number of tree nodes at each depth, per turn (turns with no node at a depth count as 0).

In [ ]:
MAX_DEPTH_SHOW = 10

def tree_depth_counts(tree):
    counts = {}
    def traverse(node, depth):
        counts[depth] = counts.get(depth, 0) + 1
        for child in node[1:]:
            if isinstance(child, list):
                traverse(child, depth + 1)
    for root_move in tree:
        if isinstance(root_move, list):
            traverse(root_move, 1)
    return counts

s1 = turns[turns['model_names'].map(is_open_source) & turns['move_success']].copy()
s1['model'] = s1['model_names']
turn_counts = [(m, tree_depth_counts(t)) for m, t in zip(s1['model'], s1['trees'])]
max_depth = max(d for _, c in turn_counts for d in c)

model_depth = (
    pd.DataFrame([{'model': m, 'depth': d, 'n_nodes': c.get(d, 0)}
                  for m, c in turn_counts for d in range(1, max_depth + 1)])
    .groupby(['model', 'depth'])['n_nodes'].mean().reset_index()
)
model_depth = model_depth[model_depth['depth'] <= MAX_DEPTH_SHOW]
models_sorted = model_depth.groupby('model')['n_nodes'].sum().sort_values(ascending=False).index.tolist()
depths = list(range(1, MAX_DEPTH_SHOW + 1))

ncols = 4
nrows = (len(models_sorted) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.6))
axes = axes.flatten()
for ax, model in zip(axes, models_sorted):
    sub = model_depth[model_depth['model'] == model].sort_values('depth')
    ax.bar(sub['depth'], sub['n_nodes'], color=family_color(model), alpha=0.85, width=0.6)
    ax.set_title(model, fontsize=11)
    ax.set_xlabel('Depth', fontsize=11)
    ax.set_ylabel('Mean # nodes', fontsize=11)
    ax.set_xticks(depths)
    ax.set_ylim(0, model_depth['n_nodes'].max() * 1.1)
for ax in axes[len(models_sorted):]:
    ax.set_visible(False)
plt.tight_layout()
save(fig, 'figS1_nodes_by_depth_per_model', bbox_inches='tight')
plt.show()

## Figure S2: Model recovery

Δ = (NLL_myopic − NLL_full) / N. Data simulated from the full-tree model should give Δ > 0; data simulated from the
myopic model should give Δ < 0. Models are ordered by sample size.

In [ ]:
rec = pd.DataFrame([
    {'name': r['model'], 'N': r['n_samples'],
     'delta_full': r['sim_full']['delta'], 'delta_myopic': r['sim_myopic']['delta']}
    for r in (json.load(open(RESULTS_DIR / 'recovery' / f'{m}.json')) for m in FITTED_MODELS)
]).sort_values('N').set_index('name')
rec['recovered'] = (rec['delta_full'] > 0) & (rec['delta_myopic'] < 0)
print(f"Recovered: {rec['recovered'].sum()}/{len(rec)}")

COLOR_FULL, COLOR_MYOPIC, COLOR_FAIL = '#38a2ed', '#3dee7e', '#cccccc'

def recovery_bar(ax, values, expected_positive, color, title):
    y = np.arange(len(values))
    ax.barh(y, values, height=0.6, alpha=0.85,
            color=[color if (v > 0) == expected_positive else COLOR_FAIL for v in values])
    ax.axvline(0, color='black', lw=0.8, ls='--')
    ax.set_xlabel('ΔNLL per sample\n(myopic - full-tree)', fontsize=11)
    ax.set_title(title, fontsize=11)

# Shared y labels; one legend in the empty lower right of panel 1
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True)
recovery_bar(axes[0], rec['delta_full'], True, COLOR_FULL,
             'Simulated from full-tree model\n($\\Delta > 0$: full-tree fits better)')
recovery_bar(axes[1], rec['delta_myopic'], False, COLOR_MYOPIC,
             'Simulated from myopic model\n($\\Delta < 0$: myopic fits better)')
axes[2].barh(np.arange(len(rec)), rec['N'], height=0.6, color='#888888', alpha=0.75)
axes[2].set_xlabel('Number of turns', fontsize=11)
axes[2].set_title('Sample size', fontsize=11)
for ax in axes:
    ax.tick_params(axis='y', length=0)
axes[0].set_yticks(np.arange(len(rec)))
axes[0].set_yticklabels(rec.index, fontsize=9)
axes[0].legend(handles=[mpatches.Patch(color=COLOR_FULL, alpha=0.85, label='Correct direction (full-tree)'),
                        mpatches.Patch(color=COLOR_MYOPIC, alpha=0.85, label='Correct direction (myopic)'),
                        mpatches.Patch(color=COLOR_FAIL, alpha=0.85, label='Wrong direction')],
               fontsize=8, loc='lower right', frameon=False)
plt.tight_layout()
save(fig, 'figS2_model_recovery', bbox_inches='tight')
plt.show()

## Figure S3: Normalized feature weights vs winning rate

Weights from the myopic fit, divided by the four-in-a-row weight. One PDF per panel.

In [ ]:
S3_PANELS = [
    # (column, x label, xlim, label offsets, file name)
    ('w_centre_norm', 'Centre weight (norm.)', (-0.03, 0.155),
     {'qwen3-235b-a22b-thinking': (-93, 2), 'gpt-oss-120b-medium': (6, -6)}, 'figS3A_centre_weight'),
    ('w_c2_norm', 'Connected-two-in-a-row weight (norm.)', None,
     {'qwen3-235b-a22b-thinking': (6, -4), 'qwen3-next-80b-a3b-thinking': (6, -2), 'gpt-oss-120b-medium': (6, -6)},
     'figS3B_connected_two_weight'),
    ('w_uc2_norm', 'Unconnected-two-in-a-row weight (norm.)', None,
     {'qwen3-235b-a22b-thinking': (6, -4), 'qwen3-next-80b-a3b-thinking': (6, -2), 'gpt-oss-120b-medium': (6, -6)},
     'figS3C_unconnected_two_weight'),
    ('w_three_norm', 'Three-in-a-row weight (norm.)', None,
     {'qwen3-235b-a22b-thinking': (6, -4), 'qwen3-next-80b-a3b-thinking': (6, -4), 'gpt-oss-120b-medium': (6, -6)},
     'figS3D_three_weight'),
]

for col, xlabel, xlim, offsets, fname in S3_PANELS:
    r, p = pearsonr(myopic_w[col], myopic_w['win_rate'])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(myopic_w[col], myopic_w['win_rate'], c=[family_color(m) for m in myopic_w.index],
               s=80, alpha=0.9, edgecolors='none', zorder=3)
    label_points(ax, myopic_w[col], myopic_w['win_rate'], myopic_w.index, offsets)
    if xlim:
        ax.set_xlim(*xlim)
    ax.set_ylim(0.11, 0.85)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Win rate', fontsize=11)
    ax.set_title(rf'$r$ = {r:.2f}, $p$ = {p:.3f} {sig_stars(p)}', fontsize=11)
    plt.tight_layout()
    save(fig, fname)
    plt.show()

## Figure S4: Window-count feature set

The four hand-designed pattern features are replaced by counts of every length-4 window by its (own, opponent)
piece composition (14 features, plus the centre feature), with weights fit separately for each model and variant.
As in Fig 3C/3D, models are sorted by myopic accuracy. Where the discount fit collapsed to myopic (γ < 1e-3), its
accuracy is set to the myopic accuracy.

In [ ]:
GAMMA_EPS = 1e-3

fits_wc, wide_wc = load_fits('windowcount')

# gamma ~ 0: the discount model is the myopic model, so equate their accuracy (removes argmax-tie noise)
gamma_wc = fits_wc[fits_wc['variant'] == 'gamma'].set_index('model')['gamma']
collapsed = gamma_wc[gamma_wc < GAMMA_EPS].index
wide_wc.loc[collapsed, 'accuracy_gamma'] = wide_wc.loc[collapsed, 'accuracy_myopic']

print(f"myopic NLL < full-tree NLL in {(wide_wc['nll_myopic'] < wide_wc['nll_full']).sum()}/{len(wide_wc)} models; "
      f"myopic accuracy > full-tree in {(wide_wc['accuracy_myopic'] > wide_wc['accuracy_full']).sum()}; "
      f"myopic NLL < no-tree in {(wide_wc['nll_myopic'] < wide_wc['nll_notree']).sum()}")
print('Discount fits with gamma > 1e-3:', gamma_wc[gamma_wc >= GAMMA_EPS].round(2).to_dict())
wide_wc.round(3)

In [ ]:
def variant_bars_wc(metric, xlabel, xlim=None, legend=False):
    # Same as variant_bars, on the window-count fits
    order = wide_wc.sort_values('accuracy_myopic').index.tolist()
    y = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(5.7, 4.5))
    for (variant, color), off in zip(VAR_COLORS, OFFSETS):
        ax.barh(y + off * BAR_HEIGHT, [wide_wc.loc[m, f'{metric}_{variant}'] for m in order], BAR_HEIGHT,
                label=VAR_LABELS[variant], color=color, alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(order, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=11)
    if xlim:
        ax.set_xlim(*xlim)
    if legend:
        ax.legend(loc='lower right', frameon=False, fontsize=8)
    plt.tight_layout()
    return fig

# Fig S4 (left)
fig = variant_bars_wc('nll', 'Negative log likelihood per sample')
save(fig, 'figS4_windowcount_nll', bbox_inches='tight')
plt.show()

# Fig S4 (right)
fig = variant_bars_wc('accuracy', 'Prediction accuracy', xlim=(0, 0.7))
save(fig, 'figS4_windowcount_accuracy', bbox_inches='tight')
plt.show()